# Partie C - Embeddings statiques et reseaux profonds



## 0. Environnement et regles du TP

Le decoupage officiel train/test est conserve. Le test officiel ne sert qu'a l'evaluation finale. Toutes les decisions sont prises sur la validation.

In [4]:
# Configuration simple
RANDOM_STATE = 42

import re
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

try:
    import torch
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Peripherique :', DEVICE)
    if DEVICE.type == 'cuda':
        print('GPU :', torch.cuda.get_device_name(0))
except ImportError:
    torch = None
    DEVICE = 'cpu'
    print('PyTorch indisponible : calcul CPU.')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve
)

RESULTS = []

def add_result(name, y_true, scores, threshold=0.5, split='validation'):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = (scores >= threshold).astype(int)
    row = {
        'modele': name,
        'split': split,
        'F1 sarcastique': f1_score(y_true, pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, scores),
        'Macro-F1': f1_score(y_true, pred, average='macro', zero_division=0),
        'seuil': threshold,
    }
    RESULTS.append(row)
    return row


Peripherique : cuda
GPU : NVIDIA GeForce RTX 2050


In [5]:
BASE = 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/'
full = pd.read_csv(BASE + 'training_data.csv')
test = pd.read_csv(BASE + 'testing_data.csv')
full['id'] = range(len(full))
test['id'] = range(len(test))
print('Train officiel :', full.shape)
print('Test officiel  :', test.shape)
display(full.head())


Train officiel : (12548, 5)
Test officiel  : (3000, 5)


,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4


# Partie C - Embeddings statiques et reseaux profonds

## C1, C2, C3 et C4

A implementer avec le meme budget, early stopping sur le F1 de validation, masque de padding et comparaison avec/sans encodage positionnel.

- C1 : BiLSTM + dernier etat ou max-pooling
- C2 : BiLSTM + attention `alpha_i = softmax(v.T tanh(W h_i))`
- C3 : mini-Transformer de 2 couches, 4 tetes, `d_model=128`
- C4 : char-CNN optionnel
- AraVec 100d/300d : embeddings figes puis degeles, taux OOV

In [ ]:

import copy
from collections import Counter

def _column(df, names):
    for name in names:
        if name in df.columns:
            return name
    raise KeyError("Colonne introuvable parmi " + str(names))

TEXT_COL_C = _column(full, ["text", "tweet", "content"])
SARC_COL_C = _column(full, ["sarcasm", "is_sarcastic", "label"])

def build_char_vocab(texts, min_freq=1):
    counts = Counter(ch for text in texts for ch in str(text))
    itos = ["<pad>", "<unk>"] + sorted(ch for ch, n in counts.items() if n >= min_freq)
    return {ch: i for i, ch in enumerate(itos)}, itos

def oov_rate(texts, vocab):
    chars = [ch for text in texts for ch in str(text)]
    return sum(ch not in vocab for ch in chars) / max(1, len(chars))

def encode_texts(texts, vocab, max_length=64):
    out = np.zeros((len(texts), max_length), dtype="int64")
    lengths = []
    for row, text in enumerate(texts):
        ids = [vocab.get(ch, 1) for ch in str(text)[:max_length]]
        out[row, :len(ids)] = ids
        lengths.append(max(1, len(ids)))
    return out, np.asarray(lengths)

class _TextDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x, self.y = torch.as_tensor(x), torch.as_tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

class BiLSTMClassifier(torch.nn.Module):
    def __init__(self, vocab_size, dim=32, hidden=32, attention=False):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, dim, padding_idx=0)
        self.lstm = torch.nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.attention = attention
        self.score = torch.nn.Linear(hidden * 2, 1)
        self.classifier = torch.nn.Linear(hidden * 2, 1)
    def forward(self, ids, return_attention=False):
        x, _ = self.lstm(self.embedding(ids))
        mask = ids.ne(0)
        if self.attention:
            weights = self.score(x).squeeze(-1).masked_fill(~mask, -1e4)
            weights = torch.softmax(weights, dim=1)
            pooled = (x * weights.unsqueeze(-1)).sum(1)
        else:
            weights = None
            pooled = (x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        result = self.classifier(pooled).squeeze(-1)
        return (result, weights) if return_attention else result

class TransformerClassifier(torch.nn.Module):
    def __init__(self, vocab_size, dim=32, heads=4, layers=1, max_length=64,
                 positional=True):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, dim, padding_idx=0)
        self.positional = torch.nn.Parameter(torch.zeros(1, max_length, dim),
                                             requires_grad=positional)
        layer = torch.nn.TransformerEncoderLayer(dim, heads, batch_first=True,
                                                 dropout=0.1)
        self.encoder = torch.nn.TransformerEncoder(layer, layers)
        self.classifier = torch.nn.Linear(dim, 1)
    def forward(self, ids):
        mask = ids.eq(0)
        x = self.encoder(self.embedding(ids) + self.positional[:, :ids.size(1)],
                         src_key_padding_mask=mask)
        valid = (~mask).unsqueeze(-1)
        pooled = (x * valid).sum(1) / valid.sum(1).clamp_min(1)
        return self.classifier(pooled).squeeze(-1)

def fit_torch_classifier(model, train_loader, valid_loader, epochs=2, patience=1):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    best, best_f1, stale = None, -1.0, 0
    for _ in range(epochs):
        model.train()
        for ids, y in train_loader:
            optimizer.zero_grad()
            loss_fn(model(ids.to(DEVICE)), y.to(DEVICE)).backward()
            optimizer.step()
        model.eval(); scores, truth = [], []
        with torch.no_grad():
            for ids, y in valid_loader:
                scores.extend(torch.sigmoid(model(ids.to(DEVICE))).cpu().numpy())
                truth.extend(y.numpy())
        current = f1_score(truth, np.asarray(scores) >= .5, zero_division=0)
        if current > best_f1:
            best_f1, best, stale = current, copy.deepcopy(model.state_dict()), 0
            stale += 1
            if stale >= patience:
                break
    if best is not None:
        model.load_state_dict(best)
    return model, best_f1

def run_partie_c():
    if torch is None:
        print("C : PyTorch est requis, expérience non exécutée.")
        return
    tr, va = train_test_split(full, test_size=.2, stratify=full[SARC_COL_C],
                              random_state=RANDOM_STATE)
    tr = tr.sample(n=min(4000, len(tr)), random_state=RANDOM_STATE)
    va = va.sample(n=min(1000, len(va)), random_state=RANDOM_STATE)
    vocab, itos = build_char_vocab(tr[TEXT_COL_C])
    xtr, _ = encode_texts(tr[TEXT_COL_C], vocab)
    xva, _ = encode_texts(va[TEXT_COL_C], vocab)
    train_loader = torch.utils.data.DataLoader(_TextDataset(xtr, tr[SARC_COL_C].astype(int).to_numpy()),
                                                batch_size=64, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(_TextDataset(xva, va[SARC_COL_C].astype(int).to_numpy()),
                                                batch_size=128)
    for name, model in [
        ("C1 BiLSTM", BiLSTMClassifier(len(itos))),
        ("C2 BiLSTM attention", BiLSTMClassifier(len(itos), attention=True)),
        ("C3 Transformer position", TransformerClassifier(len(itos), positional=True)),
        ("C3 Transformer sans position", TransformerClassifier(len(itos), positional=False)),
    ]:
        model, _ = fit_torch_classifier(model, train_loader, valid_loader)
        model.eval(); raw = []
        with torch.no_grad():
            for ids, _ in valid_loader:
                raw.extend(torch.sigmoid(model(ids.to(DEVICE))).cpu().numpy())
        add_result(name, va[SARC_COL_C], raw)
    print("Vocabulaire entraînement :", len(itos), "tokens; OOV id = 1")

def try_load_aravec(path):
    path = Path(path)
    if not path.exists():
        print("AraVec optionnel : chemin absent, chargement ignoré ->", path)
        return None
    try:
        from gensim.models import KeyedVectors
        return KeyedVectors.load_word2vec_format(str(path), binary=False)
    except Exception as exc:
        print("AraVec optionnel indisponible :", exc)
        return None

run_partie_c()


Vocabulaire entraînement : 506 tokens; OOV id = 1
